# 01 · Attention atlas: where does each head look?

In a vanilla ViT, attention is the only way information moves between places. A head's **attention pattern** says, for every query token, how much it reads from every other token. This notebook does two things:

* builds an **atlas** of all 48 heads: how far they look, in which direction, and how focused they are;
* plots single patterns for chosen query points.

In the atlas, **dx < 0** means the head reads from the **west**. In midlatitude westerlies that is *upstream*, where tomorrow's weather comes from.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
try:
    import vit_interp
except ImportError:  # not installed: use this checkout, $VIT_INTERP_REPO, or the shared one
    for repo in (Path.cwd().parent, Path(os.environ.get("VIT_INTERP_REPO", "/home/ac.ngmahfouz/vit-interp"))):
        if (repo / "vit_interp").is_dir():
            sys.path.insert(0, str(repo)); break
import numpy as np, torch, matplotlib.pyplot as plt
from vit_interp import playground as pg
em, meta = pg.load()      # or pg.load("/path/to/checkpoint.pt"); $PLAYGROUND_CKPT also works
FAST = pg.FAST            # PLAYGROUND_FAST=1 -> small settings (CPU tests)

In [ ]:
xs = torch.cat([pg.state(t) for t in (["2010-01-15"] if FAST else ["2010-01-15", "2010-04-15", "2010-07-15", "2010-10-15"])])
atlas = pg.head_atlas(em, xs)            # one row per (layer, head, latitude band)
atlas.round(2).head(12)

A heat map per statistic, for the Northern Hemisphere midlatitudes (30–60°N):

In [ ]:
band = "NH mid"
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (stat, cmap) in zip(axes, [("dx", "RdBu_r"), ("dy", "RdBu_r"), ("dist", "viridis"), ("entropy", "magma")]):
    t = atlas[atlas.band == band].pivot(index="layer", columns="head", values=stat)
    v = np.abs(t.values).max()
    im = ax.imshow(t.values, cmap=cmap, **({"vmin": -v, "vmax": v} if cmap == "RdBu_r" else {}))
    for (i, j), val in np.ndenumerate(t.values):
        ax.text(j, i, f"{val:.0f}" if stat != "entropy" else f"{val:.1f}", ha="center", va="center", fontsize=7)
    ax.set_title(f"{stat} ({band})"); ax.set_xlabel("head"); ax.set_ylabel("layer"); plt.colorbar(im, ax=ax)
fig.tight_layout()

Heads that look **upstream** (most negative dx) in the NH midlatitudes, and the most **local** ones:

In [ ]:
nh = atlas[atlas.band == "NH mid"]
display(nh.nsmallest(5, "dx")[["layer", "head", "dx", "dy", "dist", "entropy"]].round(2))
display(nh.nsmallest(5, "dist")[["layer", "head", "dx", "dy", "dist", "entropy"]].round(2))

## One head, one query point

In [ ]:
west = nh.nsmallest(1, "dx").iloc[0]
L, H = int(west["layer"]), int(west["head"])
queries = {"North Atlantic (50N, 30W)": (50, 330), "Tropical Pacific (0, 180)": (0, 180),
           "Southern Ocean (55S, 90E)": (-55, 90), "Near the North Pole (85N, 0)": (85, 0)}
for title, q in queries.items():
    a = pg.attention(em, xs[:1], L, H, q)
    pg.plot_tokens(a, f"L{L}H{H} from {title}", mark=q, log=True)

Is the pattern set by **position** (the learned embedding) or by **content** (today's weather)? Compare the same head and query on two dates:

In [ ]:
q = (50, 330)
a1 = pg.attention(em, pg.state("2010-01-15"), L, H, q)
a2 = pg.attention(em, pg.state("2010-01-20"), L, H, q)
pg.plot_maps({"Jan 15": np.log10(a1 + 1e-6), "Jan 20": np.log10(a2 + 1e-6)}, symmetric=False, cmap="viridis")
print("overlap (sum of min):", float(np.minimum(a1, a2).sum()))

## Try this

1. Layer 0 heads have entropy close to log(4050) ≈ 8.3, so they spread attention almost uniformly. What could a global-average head be for? (Hint: global means of mass and energy, or the time of day through SOLIN.)
2. Find a head whose dx flips sign between the NH and SH midlatitudes, or between the midlatitudes and the tropics. Do the tropical easterlies show up as heads looking **east**?
3. Does an upstream-looking head look **farther** west in winter, when the jet is faster? Split `xs` into January and July and compare `dx`. If it does, the head is advecting with the flow, not looking at a fixed offset.
4. Plot `dy` near the poles. Do heads wrap across the pole, or treat the grid as a flat rectangle?
5. Pick a head and a query, and compute the attention-weighted mean wind at the tokens it reads. Is the displacement consistent with U × 6 h?